# Atelier 4 — Partie 1 : Auto-Encodeur (AE) et Variational Auto-Encodeur (VAE)

**Dataset** : [MNIST](https://www.kaggle.com/datasets/hojjatk/mnist-dataset) (téléchargé automatiquement via `torchvision.datasets.MNIST`).

Mode d'exécution : Google Colab, GPU Tesla T4, graine fixée à **42**.

Objectifs :
1. Établir une architecture d'**auto-encodeur** et l'entraîner sur MNIST.
2. Établir une architecture de **VAE** et l'entraîner sur MNIST.
3. Évaluer les deux modèles (Loss, KL divergence, reconstructions).
4. Tracer l'**espace latent** des deux modèles et conclure.

## 1. Setup et hyperparamètres

In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

# Hyperparametres communs
LATENT_DIM = 2          # espace latent 2D pour tracer directement
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3


## 2. Données MNIST

In [ ]:
transform = transforms.ToTensor()  # valeurs dans [0,1]

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Train :", len(train_ds), " | Test :", len(test_ds))
# apercu
imgs, labels = next(iter(train_loader))
fig, ax = plt.subplots(1, 8, figsize=(10,2))
for i in range(8):
    ax[i].imshow(imgs[i,0], cmap="gray"); ax[i].axis("off"); ax[i].set_title(int(labels[i]))
plt.suptitle("Echantillons MNIST")
plt.show()


## 3. Architecture de l'Auto-Encodeur (AE)

Encodeur convolutif : 28×28 → 14×14 → 7×7, puis projection vers l'espace latent de dimension 2.
Décodeur symétrique avec `ConvTranspose2d`.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),   # 28 -> 14
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 14 -> 7
            nn.ReLU(inplace=True),
        )
        self.fc = nn.Linear(64*7*7, latent_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.fc(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64*7*7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),  # 7 -> 14
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),   # 14 -> 28
            nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 64, 7, 7)
        return self.deconv(h)

class AutoEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

ae = AutoEncoder().to(device)
print(ae)
print("Parametres AE :", sum(p.numel() for p in ae.parameters()))


## 4. Entraînement de l'AE

In [ ]:
def train_ae(model, loader, epochs=EPOCHS, lr=LR):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    history = []
    model.train()
    for ep in range(1, epochs+1):
        t0 = time.time()
        tot = 0.0
        for x, _ in loader:
            x = x.to(device)
            opt.zero_grad()
            x_hat, _ = model(x)
            loss = crit(x_hat, x)
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        epoch_loss = tot / len(loader.dataset)
        history.append(epoch_loss)
        print(f"AE epoch {ep:02d}/{epochs} - MSE: {epoch_loss:.5f} - {time.time()-t0:.1f}s")
    return history

ae_history = train_ae(ae, train_loader)
plt.plot(ae_history, marker="o"); plt.title("AE - Loss (MSE) par epoch"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.grid(True); plt.show()


## 5. Architecture du VAE

L'encodeur produit μ et log σ² ; on échantillonne z via la reparamétrisation z = μ + σ·ε.
La loss combine la reconstruction et la **KL divergence** entre q(z|x) et p(z)=N(0,I).

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(64*7*7, latent_dim)
        self.fc_logvar = nn.Linear(64*7*7, latent_dim)
    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar, z

vae = VAE().to(device)
print("Parametres VAE :", sum(p.numel() for p in vae.parameters()))


In [ ]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    # reconstruction (MSE) + beta * KL
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta*kl, recon, kl

def train_vae(model, loader, epochs=EPOCHS, lr=LR, beta=1.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"total": [], "recon": [], "kl": []}
    model.train()
    for ep in range(1, epochs+1):
        t0 = time.time()
        tot_t = tot_r = tot_k = 0.0
        for x, _ in loader:
            x = x.to(device)
            opt.zero_grad()
            x_hat, mu, logvar, _ = model(x)
            loss, recon, kl = vae_loss(x_hat, x, mu, logvar, beta=beta)
            loss.backward()
            opt.step()
            n = x.size(0)
            tot_t += loss.item()*n; tot_r += recon.item()*n; tot_k += kl.item()*n
        N = len(loader.dataset)
        hist["total"].append(tot_t/N); hist["recon"].append(tot_r/N); hist["kl"].append(tot_k/N)
        print(f"VAE epoch {ep:02d}/{epochs} - total: {hist['total'][-1]:.5f} recon: {hist['recon'][-1]:.5f} KL: {hist['kl'][-1]:.5f} - {time.time()-t0:.1f}s")
    return hist

vae_history = train_vae(vae, train_loader)


## 6. Courbes de loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(ae_history, marker="o", label="AE (MSE)")
axes[0].set_title("AE - Loss par epoch"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].grid(True); axes[0].legend()

axes[1].plot(vae_history["total"], marker="o", label="Total")
axes[1].plot(vae_history["recon"], marker="s", label="Reconstruction")
axes[1].plot(vae_history["kl"], marker="^", label="KL divergence")
axes[1].set_title("VAE - Loss par epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].grid(True); axes[1].legend()
plt.tight_layout(); plt.show()


## 7. Reconstructions (original vs AE vs VAE)

In [ ]:
ae.eval(); vae.eval()
x, _ = next(iter(test_loader))
x = x.to(device)
with torch.no_grad():
    ae_hat, _ = ae(x)
    vae_hat, _, _, _ = vae(x)

n = 8
fig, axes = plt.subplots(3, n, figsize=(n*1.2, 4))
for i in range(n):
    axes[0,i].imshow(x[i,0].cpu(), cmap="gray"); axes[0,i].axis("off")
    axes[1,i].imshow(ae_hat[i,0].cpu(), cmap="gray"); axes[1,i].axis("off")
    axes[2,i].imshow(vae_hat[i,0].cpu(), cmap="gray"); axes[2,i].axis("off")
axes[0,0].set_ylabel("Original", rotation=90); 
axes[1,0].set_ylabel("AE", rotation=90);
axes[2,0].set_ylabel("VAE", rotation=90)
plt.suptitle("Reconstructions : Original / AE / VAE")
plt.tight_layout(); plt.show()


## 8. Espace latent

On encode l'ensemble de test et on trace les points z colorés par le chiffre.
- **AE** : l'espace latent n'est pas contraint ; les classes peuvent former des clusters séparés mais l'espace n'est pas continu → mauvaise génération.
- **VAE** : la KL force q(z|x) vers N(0,I) ; l'espace latent est continu et régulier → bonne génération.

In [ ]:
def encode_all(model, loader, is_vae=False):
    zs, ys = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            if is_vae:
                mu, logvar = model.encoder(x)
                z = mu
            else:
                z = model.encoder(x)
            zs.append(z.cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(zs), np.concatenate(ys)

z_ae, y_ae = encode_all(ae, test_loader, is_vae=False)
z_vae, y_vae = encode_all(vae, test_loader, is_vae=True)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
sc1 = axes[0].scatter(z_ae[:,0], z_ae[:,1], c=y_ae, cmap="tab10", s=5, alpha=0.7)
axes[0].set_title("AE - Espace latent (2D)"); axes[0].set_xlabel("z1"); axes[0].set_ylabel("z2"); axes[0].grid(True)
sc2 = axes[1].scatter(z_vae[:,0], z_vae[:,1], c=y_vae, cmap="tab10", s=5, alpha=0.7)
axes[1].set_title("VAE - Espace latent (2D)"); axes[1].set_xlabel("z1"); axes[1].set_ylabel("z2"); axes[1].grid(True)
fig.colorbar(sc2, ax=axes, ticks=range(10))
plt.show()


## 9. Génération de nouveaux chiffres (VAE)

On échantillonne z ~ N(0,I) et on décode → images synthétiques.

In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=device)
    gen = vae.decoder(z).cpu()

fig, axes = plt.subplots(8, 8, figsize=(6,6))
for i in range(64):
    axes[i//8, i%8].imshow(gen[i,0], cmap="gray"); axes[i//8, i%8].axis("off")
plt.suptitle("VAE - Chiffres générés (z ~ N(0,I))")
plt.tight_layout(); plt.show()

# Grille de manifold 2D pour VAE (latent 2D)
n = 15
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)
canvas = np.zeros((28*n, 28*n))
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32, device=device)
            img = vae.decoder(z).cpu().numpy()[0,0]
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(8,8))
plt.imshow(canvas, cmap="gray"); plt.title("VAE - Manifold de l'espace latent (2D)")
plt.axis("off"); plt.show()


## 10. Conclusion

- **Loss** : l'AE minimise uniquement la reconstruction (MSE) et obtient une reconstruction légèrement meilleure ; le VAE ajoute la KL qui régularise l'espace latent au prix d'une reconstruction un peu moins nette.
- **KL divergence** : elle mesure l'écart entre q(z|x) et le prior N(0,I). Elle est nulle-ish à l'équilibre pour le VAE et force la continuité de l'espace latent.
- **Espace latent** : AE → clusters séparés, trous dans l'espace (pas de génération fiable) ; VAE → nuage continu et régulier, génération et interpolation possibles.
- **Choix** : utiliser l'AE pour la compression/reconstruction pure, le VAE pour la génération et l'interpolation dans l'espace latent.